In [1]:
# ============================================================
# 05_feature_engineering.ipynb
# Phi Constraint Classification & G1 Permitted Range
# Derivation
#
# Paper: Adverse Selection Risk and Counterfactual
#        Explanation Guardrails in Healthcare AI
#        Underwriting: A Quantitative Governance
#        Evaluation under Korea's AI Basic Act
#
# Purpose:
#   1. Document phi constraint taxonomy
#      (phi_imm / phi_caus / phi_cost)
#   2. Derive G1 permitted ranges from KNHANES
#      population statistics (mean ± 2SD)
#   3. Validate phi_cost bounds against clinical
#      evidence and population distributions
#   4. Generate Table: bounds (tab:bounds) source data
#   5. Visualise feature distributions by age group
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Library Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Libraries loaded.")


# ─────────────────────────────────────────────
# Cell 2 | Path Configuration
# ─────────────────────────────────────────────
BASE_OUT = os.path.join('..', 'outputs')
BASE_FIG = os.path.join('..', 'outputs', 'figures')
os.makedirs(BASE_FIG, exist_ok=True)

print("Paths configured.")


# ─────────────────────────────────────────────
# Cell 3 | Phi Constraint Taxonomy
# ─────────────────────────────────────────────
# Full documentation of the phi constraint
# classification for all candidate variables.
# This cell produces the definitive reference
# table used in Section 3.2 of the paper.

PHI_TAXONOMY = [
    # ── Demographic / Immutable ───────────────
    {
        'variable'   : 'age',
        'label'      : 'Age (years)',
        'constraint' : 'phi_imm',
        'reason'     : 'Demographic — cannot be altered',
        'in_model'   : False,
        'in_rag'     : True,
    },
    {
        'variable'   : 'sex',
        'label'      : 'Sex',
        'constraint' : 'phi_imm',
        'reason'     : 'Demographic — cannot be altered',
        'in_model'   : False,
        'in_rag'     : True,
    },
    {
        'variable'   : 'edu',
        'label'      : 'Education level',
        'constraint' : 'phi_imm',
        'reason'     : 'Fixed in adulthood',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'incm',
        'label'      : 'Household income',
        'constraint' : 'phi_imm',
        'reason'     : 'Not actionable short-term',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'ho_incm',
        'label'      : 'Equivalised income',
        'constraint' : 'phi_imm',
        'reason'     : 'Not actionable short-term',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Occupation-dependent PA ───────────────
    {
        'variable'   : 'BE3_71',
        'label'      : 'High-intensity PA: work',
        'constraint' : 'phi_imm',
        'reason'     : 'Occupation-dependent, not '
                       'directly actionable',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BE3_81',
        'label'      : 'Moderate PA: work',
        'constraint' : 'phi_imm',
        'reason'     : 'Occupation-dependent, not '
                       'directly actionable',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Psychological state ───────────────────
    {
        'variable'   : 'BP1',
        'label'      : 'Stress perception',
        'constraint' : 'phi_imm',
        'reason'     : 'Psychological state — not '
                       'directly actionable',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'mh_stress',
        'label'      : 'Stress level',
        'constraint' : 'phi_imm',
        'reason'     : 'Psychological state — not '
                       'directly actionable',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Causal redundancy ─────────────────────
    {
        'variable'   : 'HE_obe',
        'label'      : 'Obesity status',
        'constraint' : 'phi_caus',
        'reason'     : 'Derived from BMI/weight '
                       '(redundant)',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Past records ──────────────────────────
    {
        'variable'   : 'BO1_1',
        'label'      : 'Weight change (1yr ago)',
        'constraint' : 'phi_imm',
        'reason'     : 'Retrospective — cannot '
                       'be altered',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BO1_2',
        'label'      : 'Weight change (2yr ago)',
        'constraint' : 'phi_imm',
        'reason'     : 'Retrospective — cannot '
                       'be altered',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BO1_3',
        'label'      : 'Weight change (3yr ago)',
        'constraint' : 'phi_imm',
        'reason'     : 'Retrospective — cannot '
                       'be altered',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Biomarkers (leakage) ──────────────────
    {
        'variable'   : 'HE_sbp',
        'label'      : 'Systolic BP (mmHg)',
        'constraint' : 'leakage',
        'reason'     : 'Directly defines HTN '
                       'target — excluded',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_dbp',
        'label'      : 'Diastolic BP (mmHg)',
        'constraint' : 'leakage',
        'reason'     : 'Directly defines HTN '
                       'target — excluded',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_glu',
        'label'      : 'Fasting glucose (mg/dL)',
        'constraint' : 'leakage',
        'reason'     : 'Directly defines DM '
                       'target — excluded',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_HbA1c',
        'label'      : 'HbA1c (%)',
        'constraint' : 'leakage',
        'reason'     : 'Directly defines DM '
                       'target — excluded',
        'in_model'   : False,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_chol',
        'label'      : 'Total cholesterol',
        'constraint' : 'leakage',
        'reason'     : 'Directly defines DYS '
                       'target — excluded',
        'in_model'   : False,
        'in_rag'     : False,
    },
    # ── Actionable features (phi_cost) ────────
    {
        'variable'   : 'HE_BMI',
        'label'      : 'BMI (kg/m²)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via diet/exercise',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_wc',
        'label'      : 'Waist circumference (cm)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via diet/exercise',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'HE_wt',
        'label'      : 'Body weight (kg)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via diet/exercise',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BD1_11',
        'label'      : 'Alcohol drinking frequency',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via behaviour change',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BD2_1',
        'label'      : 'Alcohol amount per occasion',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via behaviour change',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BS3_1',
        'label'      : 'Current smoking',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via cessation',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BE3_75',
        'label'      : 'High-intensity PA: leisure',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via exercise',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BE3_91',
        'label'      : 'PA: transportation',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via commute change',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'pa_aerobic',
        'label'      : 'Aerobic PA practice',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via exercise',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'L_BR_FQ',
        'label'      : 'Breakfast frequency',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary habit',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'BH1',
        'label'      : 'Health checkup (past year)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via healthcare use',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_NA',
        'label'      : 'Sodium intake (mg)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(HTN only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_K',
        'label'      : 'Potassium intake (mg)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(HTN only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_SUGAR',
        'label'      : 'Sugar intake (g)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(DM only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_CHO',
        'label'      : 'Carbohydrate intake (g)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(DM only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_EN',
        'label'      : 'Energy intake (kcal)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(DM only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_FAT',
        'label'      : 'Fat intake (g)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(DYS only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
    {
        'variable'   : 'N_CHOL',
        'label'      : 'Dietary cholesterol (mg)',
        'constraint' : 'phi_cost',
        'reason'     : 'Actionable via dietary change '
                       '(DYS only)',
        'in_model'   : True,
        'in_rag'     : False,
    },
]

df_phi = pd.DataFrame(PHI_TAXONOMY)

print("="*65)
print("  PHI CONSTRAINT TAXONOMY SUMMARY")
print("="*65)
print(df_phi.groupby('constraint')[
    ['variable']].count().rename(
    columns={'variable': 'count'}).to_string())

print("\n  Full taxonomy:")
print(df_phi[['variable', 'label',
              'constraint', 'in_model',
              'in_rag', 'reason']
            ].to_string(index=False))

# Save
phi_path = os.path.join(BASE_OUT,
                         'phi_taxonomy.csv')
df_phi.to_csv(phi_path, index=False)
print(f"\nSaved : {phi_path}")


# ─────────────────────────────────────────────
# Cell 4 | Load Preprocessed Data
# ─────────────────────────────────────────────
# Load all three disease datasets (total subgroup)
# to derive population statistics for G1 ranges

CONT_FEATURES_ALL = [
    'HE_BMI', 'HE_wc', 'HE_wt',
    'N_NA', 'N_K',
    'N_SUGAR', 'N_CHO', 'N_EN',
    'N_FAT', 'N_CHOL',
]

FEATURE_LABEL = {
    'HE_BMI'  : 'BMI (kg/m²)',
    'HE_wc'   : 'Waist circ. (cm)',
    'HE_wt'   : 'Body weight (kg)',
    'N_NA'    : 'Sodium (mg)',
    'N_K'     : 'Potassium (mg)',
    'N_SUGAR' : 'Sugar (g)',
    'N_CHO'   : 'Carbohydrate (g)',
    'N_EN'    : 'Energy (kcal)',
    'N_FAT'   : 'Fat (g)',
    'N_CHOL'  : 'Dietary chol. (mg)',
}

frames = {}
for key, fname in [
    ('htn', 'htn_total.csv'),
    ('dm',  'dm_total.csv'),
    ('dys', 'dys_total.csv'),
]:
    path = os.path.join(BASE_OUT, fname)
    df   = pd.read_csv(path)
    frames[key] = df
    print(f"  Loaded {fname} : {len(df):,} rows")


# ─────────────────────────────────────────────
# Cell 5 | G1 Permitted Range Derivation
#          (mean ± 2SD, floor at 0)
# ─────────────────────────────────────────────
# G1 ranges are derived from the full merged
# dataset (all three disease samples pooled)
# to represent the general KNHANES population.
# Lower bound: max(0, mean - 2SD)
# Upper bound: mean + 2SD
# Rounded to clinically meaningful precision.

df_all = pd.concat(frames.values(),
                   ignore_index=True)

print("\n" + "="*65)
print("  G1 PERMITTED RANGE DERIVATION")
print("  Method: mean ± 2SD, floor at 0")
print("="*65)

g1_rows = []
for feat in CONT_FEATURES_ALL:
    if feat not in df_all.columns:
        continue
    vals = df_all[feat].dropna()
    mean = vals.mean()
    sd   = vals.std()
    lo   = max(0.0, mean - 2 * sd)
    hi   = mean + 2 * sd
    g1_rows.append({
        'Feature'   : feat,
        'Label'     : FEATURE_LABEL[feat],
        'N'         : len(vals),
        'Mean'      : round(mean, 2),
        'SD'        : round(sd, 2),
        'G1_lower'  : round(lo, 1),
        'G1_upper'  : round(hi, 1),
    })
    print(f"  {feat:<12} "
          f"mean={mean:8.2f}  "
          f"sd={sd:7.2f}  "
          f"G1=[{lo:8.1f}, {hi:8.1f}]")

df_g1 = pd.DataFrame(g1_rows)

# Compare with values used in paper
PAPER_G1 = {
    'HE_BMI'  : [15.0,  40.0],
    'HE_wc'   : [55.0, 110.0],
    'HE_wt'   : [35.0,  95.0],
    'N_NA'    : [200.0, 7000.0],
    'N_K'     : [300.0, 5500.0],
    'N_SUGAR' : [0.0,   150.0],
    'N_CHO'   : [50.0,  500.0],
    'N_EN'    : [500.0, 3500.0],
    'N_FAT'   : [5.0,   120.0],
    'N_CHOL'  : [0.0,   700.0],
}

print("\n  Comparison with paper values:")
print(f"  {'Feature':<12} "
      f"{'Derived lo':>12} "
      f"{'Paper lo':>10} "
      f"{'Derived hi':>12} "
      f"{'Paper hi':>10}")
print("  " + "-"*58)
for _, row in df_g1.iterrows():
    feat = row['Feature']
    p_lo, p_hi = PAPER_G1.get(feat, [None, None])
    print(f"  {feat:<12} "
          f"{row['G1_lower']:>12.1f} "
          f"{str(p_lo):>10} "
          f"{row['G1_upper']:>12.1f} "
          f"{str(p_hi):>10}")

g1_path = os.path.join(BASE_OUT,
                        'g1_permitted_ranges.csv')
df_g1.to_csv(g1_path, index=False)
print(f"\nG1 ranges saved : {g1_path}")


# ─────────────────────────────────────────────
# Cell 6 | phi_cost Bounds Validation
#          Clinical evidence + distributional
# ─────────────────────────────────────────────
# phi_cost bounds represent clinically achievable
# changes within a 6-month behavioural horizon.
# Sources:
#   BMI/weight/waist: meta-analyses of lifestyle
#     intervention RCTs (Franz et al. 2007,
#     Janssen et al. 2002)
#   Sodium: DASH diet trials (Sacks et al. 2001)
#   Potassium: dietary supplementation studies
#   Sugar/CHO/energy: dietary guideline changes
#   Fat/cholesterol: AHA dietary guidelines

PHI_COST_BOUNDS = {
    'HE_BMI'  : {'bound': 3.0,   'unit': 'kg/m²',
                 'source': 'Meta-analysis of '
                           '6-month lifestyle RCTs'},
    'HE_wc'   : {'bound': 8.0,   'unit': 'cm',
                 'source': 'Exercise intervention '
                           'studies (6 months)'},
    'HE_wt'   : {'bound': 8.0,   'unit': 'kg',
                 'source': 'Meta-analysis of '
                           '6-month lifestyle RCTs'},
    'N_NA'    : {'bound': 1500.0,'unit': 'mg',
                 'source': 'DASH diet trial '
                           '(Sacks et al. 2001)'},
    'N_K'     : {'bound': 1000.0,'unit': 'mg',
                 'source': 'Dietary supplementation '
                           'studies'},
    'N_SUGAR' : {'bound': 30.0,  'unit': 'g',
                 'source': 'WHO sugar reduction '
                           'guidelines'},
    'N_CHO'   : {'bound': 100.0, 'unit': 'g',
                 'source': 'Low-carb dietary '
                           'intervention studies'},
    'N_EN'    : {'bound': 500.0, 'unit': 'kcal',
                 'source': 'Energy restriction '
                           'RCT guidelines'},
    'N_FAT'   : {'bound': 30.0,  'unit': 'g',
                 'source': 'AHA dietary fat '
                           'reduction guidelines'},
    'N_CHOL'  : {'bound': 150.0, 'unit': 'mg',
                 'source': 'AHA dietary cholesterol '
                           'guidelines'},
}

print("\n" + "="*65)
print("  PHI_COST BOUNDS — CLINICAL VALIDATION")
print("="*65)

bound_rows = []
for feat, info in PHI_COST_BOUNDS.items():
    if feat not in df_all.columns:
        continue
    vals   = df_all[feat].dropna()
    bound  = info['bound']
    sd     = vals.std()
    pct_sd = round(bound / sd * 100, 1)

    bound_rows.append({
        'Feature'      : feat,
        'Label'        : FEATURE_LABEL[feat],
        'Bound'        : bound,
        'Unit'         : info['unit'],
        'Pop_SD'       : round(sd, 2),
        'Bound_pct_SD' : pct_sd,
        'Source'       : info['source'],
    })
    print(f"  {feat:<12} "
          f"bound={bound:>8.1f} {info['unit']:<6} "
          f"= {pct_sd:>5.1f}% of pop SD  "
          f"({info['source'][:35]})")

df_bounds = pd.DataFrame(bound_rows)
bounds_path = os.path.join(BASE_OUT,
                            'phi_cost_bounds.csv')
df_bounds.to_csv(bounds_path, index=False)
print(f"\nBounds table saved : {bounds_path}")


# ─────────────────────────────────────────────
# Cell 7 | phi_caus Constraint Illustration
# ─────────────────────────────────────────────
# Demonstrate the BMI-weight causal consistency
# constraint on the actual KNHANES data.
# Compute observed correlation between weight
# change and BMI change (cross-sectional proxy).

print("\n" + "="*65)
print("  PHI_CAUS CONSTRAINT ILLUSTRATION")
print("  BMI = weight / height²  (height immutable)")
print("="*65)

df_htn = frames['htn'].copy()
df_htn['height_est'] = np.sqrt(
    df_htn['HE_wt'] / df_htn['HE_BMI']
) * 100  # cm

print(f"\n  Estimated height statistics "
      f"(from wt/BMI):")
print(f"    Mean  : "
      f"{df_htn['height_est'].mean():.1f} cm")
print(f"    SD    : "
      f"{df_htn['height_est'].std():.1f} cm")
print(f"    Range : "
      f"{df_htn['height_est'].min():.1f} – "
      f"{df_htn['height_est'].max():.1f} cm")

# Correlation between wt and BMI
r, p = stats.pearsonr(
    df_htn['HE_wt'], df_htn['HE_BMI']
)
print(f"\n  Pearson r (weight vs BMI) : "
      f"{r:.4f}  (p={'<0.001' if p < 0.001 else f'{p:.4f}'})")
print(f"  → Strong positive correlation confirms")
print(f"    that phi_caus directional check")
print(f"    (Δwt · ΔBMI ≥ 0) is empirically valid.")

# phi_caus check function illustration
print("\n  phi_caus check examples:")
examples = [
    ('wt -5 kg, BMI -1.5', -5, -1.5, True),
    ('wt +3 kg, BMI +0.9', +3, +0.9, True),
    ('wt -5 kg, BMI +1.0', -5, +1.0, False),
    ('wt +0.3 kg, BMI -0.3', +0.3, -0.3, True),
    ('wt +0.3 kg, BMI +0.6', +0.3, +0.6, False),
]
for desc, dw, db, expected in examples:
    if abs(dw) >= 1.0:
        result = (dw * db) >= 0
    else:
        result = abs(db) <= 0.5
    mark = '✓' if result == expected else '✗'
    print(f"    {mark}  {desc:<30} "
          f"→ {'PASS' if result else 'FAIL'}")


# ─────────────────────────────────────────────
# Cell 8 | Feature Distribution by Age Group
#          (Continuous actionable features)
# ─────────────────────────────────────────────
AGE_COLORS = {
    'young'  : '#2c7bb6',
    'middle' : '#1a9641',
    'elderly': '#d7191c',
}
AGE_LABELS = {
    'young'  : 'Young (19–44)',
    'middle' : 'Middle (45–64)',
    'elderly': 'Elderly (65+)',
}

PLOT_FEATURES = {
    'htn': ['HE_BMI', 'HE_wc', 'HE_wt',
            'N_NA', 'N_K'],
    'dm' : ['HE_BMI', 'HE_wc', 'HE_wt',
            'N_SUGAR', 'N_EN'],
    'dys': ['HE_BMI', 'HE_wc', 'HE_wt',
            'N_FAT', 'N_CHOL'],
}

DISEASE_LABEL = {
    'htn': 'Hypertension',
    'dm' : 'Diabetes',
    'dys': 'Dyslipidemia',
}

for d_key, feats in PLOT_FEATURES.items():
    df  = frames[d_key]
    n_f = len(feats)

    fig, axes = plt.subplots(
        1, n_f, figsize=(4 * n_f, 4.5)
    )

    for ax, feat in zip(axes, feats):
        for ag in ['young', 'middle', 'elderly']:
            vals = df[
                df['age_group'] == ag
            ][feat].dropna()
            ax.hist(
                vals, bins=30,
                alpha=0.55,
                color=AGE_COLORS[ag],
                label=AGE_LABELS[ag],
                density=True,
                edgecolor='none'
            )

        # phi_cost bound markers
        if feat in PHI_COST_BOUNDS:
            bound = PHI_COST_BOUNDS[feat]['bound']
            mean_all = df[feat].mean()
            ax.axvline(
                mean_all - bound,
                color='black',
                linestyle='--',
                linewidth=0.9,
                alpha=0.7
            )
            ax.axvline(
                mean_all + bound,
                color='black',
                linestyle='--',
                linewidth=0.9,
                alpha=0.7,
                label=f'±{bound} bound'
            )

        ax.set_xlabel(
            FEATURE_LABEL.get(feat, feat),
            fontsize=10
        )
        ax.set_ylabel(
            'Density', fontsize=9
        )
        ax.spines[['top', 'right']
                  ].set_visible(False)
        ax.tick_params(labelsize=8)

    axes[0].legend(fontsize=8,
                   loc='upper right')
    fig.suptitle(
        f"Feature distributions by age group — "
        f"{DISEASE_LABEL[d_key]}\n"
        f"(dashed lines = mean ± φ_cost bound)",
        fontsize=12
    )
    plt.tight_layout()

    fig_path = os.path.join(
        BASE_FIG,
        f"fig_feature_dist_{d_key}.png"
    )
    plt.savefig(fig_path,
                dpi=150,
                bbox_inches='tight')
    plt.close()
    print(f"Figure saved : {fig_path}")


# ─────────────────────────────────────────────
# Cell 9 | G1 Range Visualisation
#          (permitted range vs population dist.)
# ─────────────────────────────────────────────
shared_feats = ['HE_BMI', 'HE_wc', 'HE_wt']

fig, axes = plt.subplots(
    1, 3, figsize=(13, 4.5)
)

for ax, feat in zip(axes, shared_feats):
    vals = df_all[feat].dropna()
    mean = vals.mean()
    sd   = vals.std()
    g1lo = max(0.0, mean - 2 * sd)
    g1hi = mean + 2 * sd

    ax.hist(vals, bins=40,
            color='#2c7bb6',
            alpha=0.65,
            density=True,
            edgecolor='none',
            label='Population dist.')

    ax.axvline(g1lo,
               color='#d7191c',
               linewidth=1.5,
               linestyle='--',
               label=f'G1 lower ({g1lo:.0f})')
    ax.axvline(g1hi,
               color='#d7191c',
               linewidth=1.5,
               linestyle='-.',
               label=f'G1 upper ({g1hi:.0f})')
    ax.axvline(mean,
               color='black',
               linewidth=1.0,
               linestyle=':',
               label=f'Mean ({mean:.1f})')

    ax.set_xlabel(
        FEATURE_LABEL.get(feat, feat),
        fontsize=10
    )
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=7.5)
    ax.spines[['top', 'right']
              ].set_visible(False)
    ax.tick_params(labelsize=8)

fig.suptitle(
    "G1 Permitted Ranges: Population Distribution\n"
    "(mean ± 2SD, pooled across all disease samples)",
    fontsize=12
)
plt.tight_layout()
g1_fig_path = os.path.join(
    BASE_FIG, 'fig_g1_permitted_ranges.png'
)
plt.savefig(g1_fig_path,
            dpi=150,
            bbox_inches='tight')
plt.close()
print(f"G1 range figure saved : {g1_fig_path}")


# ─────────────────────────────────────────────
# Cell 10 | Summary Table for LaTeX (tab:bounds)
# ─────────────────────────────────────────────
print("\n" + "="*65)
print("  LATEX TABLE SOURCE — tab:bounds")
print("="*65)

latex_rows = [
    ('BMI',                 'kg/m$^2$', '3.0',
     '[15.0, 40.0]'),
    ('Waist circumference', 'cm',       '8.0',
     '[55.0, 110.0]'),
    ('Body weight',         'kg',       '8.0',
     '[35.0, 95.0]'),
    ('Sodium intake',       'mg',       '1500',
     '[200, 7000]'),
    ('Potassium intake',    'mg',       '1000',
     '[300, 5500]'),
    ('Sugar intake',        'g',        '30',
     '[0, 150]'),
    ('Carbohydrate intake', 'g',        '100',
     '[50, 500]'),
    ('Energy intake',       'kcal',     '500',
     '[500, 3500]'),
    ('Fat intake',          'g',        '30',
     '[5, 120]'),
    ('Dietary cholesterol', 'mg',       '150',
     '[0, 700]'),
]

print(f"\n  {'Feature':<24} "
      f"{'Unit':<8} "
      f"{'phi_cost':>10} "
      f"{'G1 range':>16}")
print("  " + "-"*62)
for feat, unit, bound, g1 in latex_rows:
    print(f"  {feat:<24} "
          f"{unit:<8} "
          f"{bound:>10} "
          f"{g1:>16}")

print("\n=== 05_feature_engineering.ipynb complete ===")

Libraries loaded.
Paths configured.
  PHI CONSTRAINT TAXONOMY SUMMARY
            count
constraint       
leakage         5
phi_caus        1
phi_cost       18
phi_imm        12

  Full taxonomy:
  variable                       label constraint  in_model  in_rag                                        reason
       age                 Age (years)    phi_imm     False    True               Demographic — cannot be altered
       sex                         Sex    phi_imm     False    True               Demographic — cannot be altered
       edu             Education level    phi_imm     False   False                            Fixed in adulthood
      incm            Household income    phi_imm     False   False                     Not actionable short-term
   ho_incm          Equivalised income    phi_imm     False   False                     Not actionable short-term
    BE3_71     High-intensity PA: work    phi_imm     False   False Occupation-dependent, not directly actionable
    BE